# ASVspoof5 Global Logistic Baseline Excluding A12

This notebook trains one shared binary logistic-regression model for `bonafide` vs `spoof`.
It uses the same train/test subsets from the train/dev 16-system manifests, but excludes `A12` from the spoof class.


In [ ]:
import gc
import io
import json
import pickle
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')

PLAN_BASE = (
    PROJECT_ROOT
    / 'data'
    / 'datasets'
    / 'ASVspoof5_tars'
    / 'ASVspoof5_protocols'
    / 'train_dev_16_systems_outputs'
)

MANIFEST_SPECS = [
    {
        'source_partition': 'train',
        'manifest_path': PLAN_BASE / 'train' / 'selected_utterances_plan.csv',
        'tar_dir': PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_audio_train_tars',
        'tar_prefix': 'flac_T_*.tar',
    },
    {
        'source_partition': 'dev',
        'manifest_path': PLAN_BASE / 'dev' / 'selected_utterances_plan.csv',
        'tar_dir': PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_audio_dev_tars',
        'tar_prefix': 'flac_D_*.tar',
    },
]

EXCLUDED_SPOOF_SYSTEMS = {'A12'}
ALL_SYSTEMS = {f'A{i:02d}' for i in range(1, 17)}
INCLUDED_SPOOF_SYSTEMS = sorted(ALL_SYSTEMS - EXCLUDED_SPOOF_SYSTEMS)

OUT_BASE = PROJECT_ROOT / 'data' / 'models' / 'asvspoof5_train_dev_16_systems_global_excluding_a12'
OUT_BASE.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FORCE_RECOMPUTE_EMBEDDINGS = False
SAVE_PREDICTIONS = True
LOGREG_MAX_ITER = 2000
LOGREG_C = 1.0

print('DEVICE =', DEVICE)
print('Included spoof systems =', INCLUDED_SPOOF_SYSTEMS)
print('Excluded spoof systems =', sorted(EXCLUDED_SPOOF_SYSTEMS))
print('OUT_BASE =', OUT_BASE)
for spec in MANIFEST_SPECS:
    print(spec['source_partition'], 'manifest =', spec['manifest_path'], '| exists =', spec['manifest_path'].exists())
    print(spec['source_partition'], 'tar_dir =', spec['tar_dir'], '| exists =', spec['tar_dir'].exists())


In [ ]:
# ===== Model =====
redim_model = (
    torch.hub.load(
        'IDRnD/ReDimNet',
        'ReDimNet',
        model_name='b6',
        train_type='ptn',
        dataset='vox2',
    )
    .to(DEVICE)
    .eval()
)
print('Loaded ReDimNet on', DEVICE)


In [ ]:
def embed_waveform(wav: torch.Tensor, sr: int) -> np.ndarray:
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if wav.ndim == 1:
        wav = wav.unsqueeze(0)
    if wav.shape[0] > 1:
        wav = wav[:1, :]
    wav = wav.to(DEVICE)
    with torch.no_grad():
        emb = redim_model(wav)
    return emb.squeeze(0).detach().cpu().numpy().astype(np.float32)


def embed_member_from_tar(tf: tarfile.TarFile, member: tarfile.TarInfo) -> np.ndarray:
    fobj = tf.extractfile(member)
    if fobj is None:
        raise RuntimeError(f'Cannot extract member: {member.name}')
    raw = fobj.read()
    try:
        wav, sr = torchaudio.load(io.BytesIO(raw))
    except Exception:
        import tempfile
        suffix = Path(member.name).suffix or '.flac'
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=True) as tmp:
            tmp.write(raw)
            tmp.flush()
            wav, sr = torchaudio.load(tmp.name)
    return embed_waveform(wav, sr)


def extract_embeddings_for_manifest(manifest_df: pd.DataFrame, cache_npz: Path, tar_dir: Path, tar_prefix: str, force_recompute: bool = False):
    if cache_npz.exists() and not force_recompute:
        payload = np.load(cache_npz, allow_pickle=True)
        X = payload['X']
        utt_ids = payload['utt_ids'].astype(str)
        lut = pd.DataFrame({'utt_id': utt_ids, '_idx': np.arange(len(utt_ids))})
        m = manifest_df[['utt_id']].astype(str).merge(lut, on='utt_id', how='left', validate='one_to_one')
        if m['_idx'].isna().any():
            miss = m.loc[m['_idx'].isna(), 'utt_id'].tolist()[:10]
            raise RuntimeError(f'Embedding cache missing utt_ids, examples={miss}')
        return X[m['_idx'].astype(int).to_numpy()]

    tars = sorted(tar_dir.glob(tar_prefix))
    assert len(tars) > 0, f'No tar files in {tar_dir} matching {tar_prefix}'

    need = set(manifest_df['utt_id'].astype(str).tolist())
    emb_map = {}
    found = set()

    for tar_path in tars:
        print('Reading', tar_path.name)
        with tarfile.open(tar_path, 'r') as tf:
            for member in tf:
                if not member.isfile():
                    continue
                utt = Path(Path(member.name).name).stem
                if utt not in need or utt in found:
                    continue
                emb_map[utt] = embed_member_from_tar(tf, member)
                found.add(utt)
        print('Found so far:', len(found), '/', len(need))

    missing = sorted(list(need - found))
    if missing:
        raise RuntimeError(f'Missing {len(missing)} utt_ids in tar shards. examples={missing[:10]}')

    ids = manifest_df['utt_id'].astype(str).tolist()
    X = np.stack([emb_map[u] for u in ids]).astype(np.float32)
    np.savez_compressed(cache_npz, X=X, utt_ids=np.array(ids, dtype=object))
    return X


def compute_metrics(y_true, p_spoof, thr=0.5):
    y_hat = (p_spoof >= thr).astype(int)
    cm = confusion_matrix(y_true, y_hat).tolist()
    return {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'auc': float(roc_auc_score(y_true, p_spoof)) if len(np.unique(y_true)) == 2 else None,
        'confusion_matrix': cm,
        'classification_report': classification_report(y_true, y_hat, output_dict=True, zero_division=0),
    }


def validate_manifest(df: pd.DataFrame, source_partition: str):
    req_cols = {'split', 'speaker_id', 'utt_id', 'gender', 'label', 'system_id', 'sample_class', 'target_class'}
    missing = req_cols - set(df.columns)
    if missing:
        raise ValueError(f'{source_partition} manifest missing cols: {sorted(missing)}')
    if 'source_partition' in df.columns:
        bad = df.loc[~df['source_partition'].eq(source_partition)]
        if len(bad) > 0:
            raise ValueError(f'{source_partition} manifest contains mismatched source_partition rows')
    if set(df['split'].unique()) - {'train', 'test'}:
        raise ValueError(f'{source_partition} manifest split column must only contain train/test')


In [ ]:
part_frames = []
part_embeddings = []

for spec in MANIFEST_SPECS:
    source_partition = spec['source_partition']
    manifest_path = spec['manifest_path']
    tar_dir = spec['tar_dir']
    tar_prefix = spec['tar_prefix']

    print(f'===== SOURCE PARTITION: {source_partition.upper()} =====')
    assert manifest_path.exists(), f'Missing manifest: {manifest_path}'

    df = pd.read_csv(manifest_path)
    validate_manifest(df, source_partition)
    df = df[df['target_class'].eq('bonafide') | df['target_class'].isin(INCLUDED_SPOOF_SYSTEMS)].copy()
    df['source_partition'] = source_partition
    df['binary_label'] = np.where(df['target_class'].eq('bonafide'), 0, 1).astype(int)

    print('Rows kept =', len(df))
    print(df['target_class'].value_counts().sort_index())

    cache_npz = OUT_BASE / f'embeddings_{source_partition}_global_excluding_a12.npz'
    X_part = extract_embeddings_for_manifest(
        df[['utt_id']].copy(),
        cache_npz=cache_npz,
        tar_dir=tar_dir,
        tar_prefix=tar_prefix,
        force_recompute=FORCE_RECOMPUTE_EMBEDDINGS,
    )

    part_frames.append(df.reset_index(drop=True))
    part_embeddings.append(X_part)

meta_df = pd.concat(part_frames, axis=0, ignore_index=True)
X = np.concatenate(part_embeddings, axis=0)
y = meta_df['binary_label'].to_numpy()

print('Combined rows =', len(meta_df))
print('Embedding shape =', X.shape)
print('Train rows =', int(meta_df['split'].eq('train').sum()))
print('Test rows =', int(meta_df['split'].eq('test').sum()))
print('Spoof rows =', int((y == 1).sum()))
print('Bonafide rows =', int((y == 0).sum()))


In [ ]:
is_train = meta_df['split'].eq('train').to_numpy()
is_test = meta_df['split'].eq('test').to_numpy()

X_tr, y_tr = X[is_train], y[is_train]
X_te, y_te = X[is_test], y[is_test]

assert len(X_tr) > 0 and len(X_te) > 0, 'Empty train/test split'
assert set(np.unique(y_tr)) == {0, 1}, 'Train labels invalid'
assert set(np.unique(y_te)) == {0, 1}, 'Test labels invalid'

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

clf = LogisticRegression(
    max_iter=LOGREG_MAX_ITER,
    C=LOGREG_C,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_tr_s, y_tr)

p_tr = clf.predict_proba(X_tr_s)[:, 1]
p_te = clf.predict_proba(X_te_s)[:, 1]
m_tr = compute_metrics(y_tr, p_tr, thr=0.5)
m_te = compute_metrics(y_te, p_te, thr=0.5)

run_summary = {
    'task': 'bonafide_vs_spoof_global_excluding_a12',
    'included_spoof_systems': INCLUDED_SPOOF_SYSTEMS,
    'excluded_spoof_systems': sorted(EXCLUDED_SPOOF_SYSTEMS),
    'train_rows': int(is_train.sum()),
    'test_rows': int(is_test.sum()),
    'train_bonafide_rows': int(((meta_df['split'].eq('train')) & (meta_df['binary_label'].eq(0))).sum()),
    'train_spoof_rows': int(((meta_df['split'].eq('train')) & (meta_df['binary_label'].eq(1))).sum()),
    'test_bonafide_rows': int(((meta_df['split'].eq('test')) & (meta_df['binary_label'].eq(0))).sum()),
    'test_spoof_rows': int(((meta_df['split'].eq('test')) & (meta_df['binary_label'].eq(1))).sum()),
    'feature_dim': int(X.shape[1]),
    'train_acc': m_tr['accuracy'],
    'train_auc': m_tr['auc'],
    'train_cm': m_tr['confusion_matrix'],
    'test_acc': m_te['accuracy'],
    'test_auc': m_te['auc'],
    'test_cm': m_te['confusion_matrix'],
}

with open(OUT_BASE / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(OUT_BASE / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open(OUT_BASE / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(run_summary, f, indent=2)

meta_df.to_csv(OUT_BASE / 'training_manifest_used.csv', index=False)

if SAVE_PREDICTIONS:
    pred_df = meta_df[['source_partition', 'split', 'speaker_id', 'utt_id', 'gender', 'label', 'system_id', 'sample_class', 'target_class', 'binary_label']].copy()
    pred_df['p_spoof'] = np.concatenate([p_tr, p_te])
    pred_df['y_hat'] = (pred_df['p_spoof'] >= 0.5).astype(int)
    pred_df.to_csv(OUT_BASE / 'predictions.csv', index=False)

print(json.dumps(run_summary, indent=2))


In [ ]:
summary_df = pd.DataFrame([
    {
        'split': 'train',
        'accuracy': m_tr['accuracy'],
        'auc': m_tr['auc'],
        'rows': int(is_train.sum()),
    },
    {
        'split': 'test',
        'accuracy': m_te['accuracy'],
        'auc': m_te['auc'],
        'rows': int(is_test.sum()),
    },
])
display(summary_df)

plt.figure(figsize=(6, 4))
plt.bar(summary_df['split'], summary_df['accuracy'])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Global Logistic Accuracy Excluding A12')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(summary_df['split'], summary_df['auc'])
plt.ylim(0, 1)
plt.ylabel('AUC')
plt.title('Global Logistic AUC Excluding A12')
plt.tight_layout()
plt.show()
